In [2]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path(r"C:\Users\hdped\Desktop\ANTAQ_Projeto\data")
PASTAS = {
    "01_Cadastros":   BASE_DIR / "01_Cadastros",
    "02_Operacoes":   BASE_DIR / "02_Operacoes",
    "03_Indicadores": BASE_DIR / "03_Indicadores",
}
SEPARADORES = [";", ",", "\t", "|"]
ENCODINGS = ["utf-8", "latin-1", "cp1252"]

def ler_csv(filepath):
    for enc in ENCODINGS:
        for sep in SEPARADORES:
            try:
                df = pd.read_csv(filepath, sep=sep, encoding=enc, low_memory=False, nrows=50000)
                if df.shape[1] > 1:
                    return df, sep, enc
            except Exception:
                continue
    return None, "?", "?"

def avaliar_ficheiro(filepath):
    tamanho_mb = round(filepath.stat().st_size / 1024 / 1024, 2)
    df, sep, enc = ler_csv(filepath)
    if df is None:
        return {"ficheiro": filepath.name, "tamanho_MB": tamanho_mb, "status": "ERRO"}
    try:
        total_linhas = sum(1 for _ in open(filepath, encoding=enc)) - 1
    except Exception:
        total_linhas = len(df)
    colunas = list(df.columns)
    nulos = {col: round(df[col].isna().sum() / len(df) * 100, 1) for col in df.columns}
    score = round(100 - np.mean(list(nulos.values())), 1)
    return {
        "ficheiro": filepath.name,
        "tamanho_MB": tamanho_mb,
        "status": "OK",
        "sep": sep,
        "enc": enc,
        "linhas": total_linhas,
        "colunas": colunas,
        "nulos": nulos,
        "score": score,
    }

total_f = total_l = total_mb = 0
scores = []
print("=" * 70)
print("  ANTAQ PROJETO - INVENTARIO DE MATERIA PRIMA")
print("=" * 70)
for nome, caminho in PASTAS.items():
    if not caminho.exists():
        print(f"\nPasta nao encontrada: {caminho}")
        continue
    csvs = sorted(caminho.glob("*.csv"))
    print(f"\n--- {nome} ({len(csvs)} ficheiros) ---")
    for csv_path in csvs:
        print(f"  A ler: {csv_path.name}")
        info = avaliar_ficheiro(csv_path)
        total_f += 1
        total_mb += info["tamanho_MB"]
        if info["status"] == "OK":
            total_l += info["linhas"]
            scores.append(info["score"])
            print(f"  [OK] {info['ficheiro']}")
            print(f"       Linhas: {info['linhas']:,} | Colunas: {len(info['colunas'])} | Score: {info['score']}/100 | Sep: '{info['sep']}' | Enc: {info['enc']}")
            print(f"       Colunas: {', '.join(info['colunas'])}")
            cols_ruins = [c for c, v in info['nulos'].items() if v > 30]
            if cols_ruins:
                print(f"       AVISO nulos>30%: {', '.join(cols_ruins)}")
        else:
            print(f"  [ERRO] {info['ficheiro']}")
        print()

print("=" * 70)
print("  RESUMO FINAL")
print("=" * 70)
print(f"  Ficheiros : {total_f}")
print(f"  Linhas    : {total_l:,}")
print(f"  Disco     : {round(total_mb,1)} MB")
print(f"  Qualidade : {round(np.mean(scores),1)}/100")
print("=" * 70)

  ANTAQ PROJETO - INVENTARIO DE MATERIA PRIMA

--- 01_Cadastros (8 ficheiros) ---
  A ler: Relatorio_Area.csv
  [OK] Relatorio_Area.csv
       Linhas: 1,078 | Colunas: 4 | Score: 100.0/100 | Sep: ';' | Enc: utf-8
       Colunas: Nome Área, Código, Instalação/Porto, Identificação

  A ler: Relatorio_Armador_Estrangeiro.csv
  [OK] Relatorio_Armador_Estrangeiro.csv
       Linhas: 4,788 | Colunas: 5 | Score: 100.0/100 | Sep: ';' | Enc: utf-8
       Colunas: Código Armador, País, Apelido, Nome, Ativo

  A ler: Relatorio_Berco.csv
  [OK] Relatorio_Berco.csv
       Linhas: 875 | Colunas: 5 | Score: 99.7/100 | Sep: ';' | Enc: utf-8
       Colunas: Código do Berço, Nome do Berço, Porto Organizado/Instalação Portuária Autorizada, Instalação de Acostagem, Código Instalação de Acostagem

  A ler: Relatorio_Embarcação_Estrangeira.csv
  [OK] Relatorio_Embarcação_Estrangeira.csv
       Linhas: 29,850 | Colunas: 11 | Score: 88.1/100 | Sep: ';' | Enc: utf-8
       Colunas: ID Embarcação, Nome da Embarc